## Problema

El Problema del Viajante de Comercio (TSP) es uno de los problemas clásicos más importantes dentro del campo de la optimización combinatoria. Su objetivo consiste en determinar la ruta más corta posible que permita a un viajante recorrer un conjunto de ciudades, visitando cada una exactamente una vez y regresando al punto de partida. La complejidad de este problema radica en que el número de posibles rutas crece de manera exponencial conforme aumenta la cantidad de ciudades, lo que lo convierte en un problema computacionalmente desafiante.

Para abordar este tipo de problemas, se han desarrollado diversos métodos bioinspirados. Uno de los más destacados es la Optimización por Colonia de Hormigas (ACO), un algoritmo inspirado en el comportamiento de las hormigas, las cuales depositan feromonas en los caminos que recorren para señalar rutas eficientes. En este enfoque, las rutas más cortas reciben mayor cantidad de feromona, aumentando la probabilidad de ser elegidas por otras hormigas en iteraciones posteriores.

El objetivo principal en este contexto es minimizar la longitud total del recorrido. Para lograrlo, el algoritmo incorpora mecanismos como la evaporación de feromonas, lo que permite olvidar soluciones subóptimas y evitar quedar atrapado en mínimos locales. Asimismo, se mantiene la restricción fundamental de que cada ciudad debe ser visitada únicamente una vez durante el recorrido, excepto el regreso al punto de origen.

Este problema y su solución mediante ACO forman parte del área de optimización, donde también se estudian otros enfoques como la Optimización por Enjambre de Partículas (PSO), inspirada en el comportamiento colectivo de animales, y utilizada para encontrar soluciones óptimas en funciones matemáticas complejas.

In [1]:
import random
import sys
import math

In [2]:

# Genera una matriz de distancias aleatorias entre ciudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]
    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]
    return matriz

In [3]:
# Elige el siguiente paso basado en feromonas, distancias y ciudades no visitadas
def eligeCiudad(dists, ferom, visitadas):
    listaPesos = []
    disponibles = []
    actual = visitadas[-1]
    
    alfa = 1.0  # Influencia de las feromonas
    beta = 0.5  # Influencia de las distancias
    
    for i in range(len(dists)):
        if i not in visitadas:
            # Cálculo de probabilidad basado en la fórmula del algoritmo
            fer = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0 / dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)
            
    valor = random.random() * sum(listaPesos)
    acumulado = 0.0
    i = -1
    while valor > acumulado:
        i += 1
        acumulado += listaPesos[i]
    return disponibles[i]

In [7]:
# Una "hormiga" completa un recorrido completo
def eligeCamino(distancias, feromonas):
    camino = [0] # Siempre inicia en la ciudad 0
    longCamino = 0
    while len(camino) < len(distancias):
        ciudad = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)
    
    # Regreso al origen
    longCamino += distancias[camino[-1]][0]
    camino.append(0)
    return (camino, longCamino)

In [8]:
# Deposita feromonas en el camino encontrado
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i+1]] += dosis

# Reduce la intensidad de las feromonas (evaporación)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9 # Coeficiente de evaporación de 0.1

In [9]:
# Función principal que coordina el algoritmo
def hormigas(distancias, iteraciones, distMedia):
    n = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]
    mejorCamino = []
    longMejorCamino = sys.maxsize # Valor inicial infinito
    
    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)
        if longCamino <= longMejorCamino:
            mejorCamino = camino
            longMejorCamino = longCamino
            # Refuerza el rastro si es el mejor camino hasta ahora
            rastroFeromonas(feromonas, camino, distMedia/longCamino)
        evaporaFeromonas(feromonas)
    return (mejorCamino, longMejorCamino)

In [10]:
# --- Pruebas del algoritmo ---
numCiudades = 10
distanciaMaxima = 10
ciudades = matrizDistancias(numCiudades, distanciaMaxima)
iteraciones = 1000
distMedia = (numCiudades * distanciaMaxima) / 2

camino, longCamino = hormigas(ciudades, iteraciones, distMedia)

print(f"Mejor camino: {camino}")
print(f"Longitud del camino: {longCamino}")

Mejor camino: [0, 2, 7, 8, 5, 9, 1, 3, 4, 6, 0]
Longitud del camino: 6.889898885132018
